# Paper figure — cortical atlas Shannon observables

Generates the publication figure (panels a–d) from the DKT-62 GC results:

- **a** — population-mean redundancy R̄ on cortical surface
- **b** — population-mean vulnerability V̄ on cortical surface
- **c** — R̄ vs V̄ scatter across 62 atlas regions
- **d** — leave-one-out reliability (Spearman ρ per subject)

**Requirements:** `mne`, `nibabel`, `nilearn`, `adjustText`, `h5py`  
GC results are included in the repository at `results/gc_analysis/gc_dkt62_results/`. No preprocessing needed — just run this notebook.

In [ ]:
import os
import tempfile
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patheffects as pe
from matplotlib.image import imread
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.ticker import MaxNLocator
from scipy.stats import pearsonr, spearmanr
from adjustText import adjust_text

import mne
import nibabel as nib
from nilearn import plotting

%matplotlib inline

matplotlib.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 7,
    'axes.labelsize': 8,
    'axes.titlesize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.02,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'mathtext.default': 'regular',
})

## 1. Load GC results

In [ ]:
def find_project_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / 'results').exists() and (p / 'scripts').exists():
            return p
    raise RuntimeError('Could not infer project root.')

project_root = find_project_root(Path.cwd())
gc_root = project_root / 'results' / 'gc_analysis'
# Use the included results folder; fall back to the most recent
# timestamped gc_dkt62_* folder if you ran the pipeline yourself.
gc_dir_default = gc_root / 'gc_dkt62_results'
if gc_dir_default.exists():
    gc_dir = gc_dir_default
else:
    gc_dirs = sorted(d for d in gc_root.glob('gc_dkt62_*') if d.is_dir())
    if not gc_dirs:
        raise FileNotFoundError(f'No gc_dkt62_* folders in {gc_root}')
    gc_dir = gc_dirs[-1]


def decode_matlab_char(obj) -> str:
    arr = np.array(obj)
    return ''.join(chr(int(c)) for c in np.ravel(arr) if int(c) != 0)

def read_str_field(group, field, default=''):
    if field not in group:
        return default
    try:
        return decode_matlab_char(group[field])
    except Exception:
        return default

def as_scalar(ds, default=np.nan):
    try:
        return float(np.array(ds).squeeze())
    except Exception:
        return default

def as_int(ds, default=-1):
    v = as_scalar(ds, default=np.nan)
    return int(v) if np.isfinite(v) else default

with h5py.File(gc_dir / 'gc_results.mat', 'r') as hf:
    all_results = hf['all_results']
    records, labels = [], None
    for i in range(all_results.shape[1]):
        ref = all_results[0, i]
        if not ref:
            continue
        sg = hf[ref]
        if 'results' not in sg:
            continue
        sid = read_str_field(sg, 'subject_id', f'subj_{i:03d}')
        outer = hf[sg['chan_labels'][0, 0]]
        lbl = [decode_matlab_char(hf[r]) for r in np.array(outer).reshape(-1)]
        if labels is None:
            labels = lbl
        res = sg['results']
        records.append({
            'subject_id': sid,
            'duration_s': as_scalar(sg.get('duration_s', np.nan)),
            'F_joint':       np.array(res['F_joint']).reshape(-1),
            'redundancy':    np.array(res['redundancy']).reshape(-1),
            'vulnerability': np.array(res['vulnerability']).reshape(-1),
        })

R = np.vstack([r['redundancy']    for r in records])
V = np.vstack([r['vulnerability'] for r in records])
J = np.vstack([r['F_joint']       for r in records])
n_sub, n_reg = R.shape

R_mean = np.nanmean(R, axis=0)
V_mean = np.nanmean(V, axis=0)
R_sd   = np.nanstd(R,  axis=0, ddof=1)
V_sd   = np.nanstd(V,  axis=0, ddof=1)

print(f'Subjects: {n_sub},  Regions: {n_reg}')

## 2. Render cortical surface panels

Projects R̄ and V̄ onto the local MNE `fsaverage` inflated surface (8 views × 2 metrics) using the `aparc` cortical annotation for label rendering.

In [ ]:
# ── fsaverage parcellation mapping ───────────────────────────────
mne_fs = Path(mne.datasets.fetch_fsaverage(verbose=False))

# Use the fsaverage aparc annotation for cortical label rendering.
our_label_map = {name: i for i, name in enumerate(labels)}
parc = {}
for hemi, hk in [('left', 'lh'), ('right', 'rh')]:
    annot_arr, _, names_raw = nib.freesurfer.read_annot(
        mne_fs / 'label' / f'{hk}.aparc.annot')
    names = [n.decode() if isinstance(n, bytes) else n for n in names_raw]
    idx = np.full(len(annot_arr), -1, dtype=int)
    for ai, aname in enumerate(names):
        key = f'{aname}-{hk}'
        if key in our_label_map:
            idx[annot_arr == ai] = our_label_map[key]
    parc[hk] = idx

def labels_to_surf(values, hemi_key):
    idx = parc[hemi_key]
    vtx = np.full(len(idx), np.nan)
    vtx[idx >= 0] = values[idx[idx >= 0]]
    return vtx


# ── Render brain panels ───────────────────────────────────────────
def autocrop(img, bg_val=1.0, margin=1):
    gray = img.mean(axis=2) if img.ndim == 3 else img
    mask = gray < (bg_val - 0.02)
    rows, cols = np.any(mask, axis=1), np.any(mask, axis=0)
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    return img[max(0, rmin-margin):rmax+margin+1,
               max(0, cmin-margin):cmax+margin+1]

cmap_name = 'RdYlBu_r'
cmap      = plt.get_cmap(cmap_name)
r_vmin, r_vmax = float(R_mean.min()), float(R_mean.max())
v_vmin, v_vmax = float(V_mean.min()), float(V_mean.max())

views = [('left','lateral'), ('left','medial'),
         ('right','medial'), ('right','lateral')]

with tempfile.TemporaryDirectory() as tmpdir:
    panel_imgs = {}
    for mk, vals, vmin, vmax in [('R', R_mean, r_vmin, r_vmax),
                                  ('V', V_mean, v_vmin, v_vmax)]:
        for vi, (hemi, view) in enumerate(views):
            hk = 'lh' if hemi == 'left' else 'rh'
            fig_b = plotting.plot_surf_stat_map(
                mne_fs / 'surf' / f'{hk}.inflated', labels_to_surf(vals, hk),
                hemi=hemi, view=view,
                bg_map=mne_fs / 'surf' / f'{hk}.sulc',
                bg_on_data=True, colorbar=False,
                cmap=cmap_name, vmin=vmin, vmax=vmax,
                threshold=None, title=None, engine='matplotlib')
            fp = os.path.join(tmpdir, f'{mk}_{vi}.png')
            fig_b.savefig(fp, dpi=250, bbox_inches='tight',
                          facecolor='white', pad_inches=0)
            plt.close(fig_b)
            panel_imgs[(mk, vi)] = autocrop(imread(fp))

    def pad_row(imgs_dict, key):
        imgs = [imgs_dict[(key, vi)] for vi in range(4)]
        H = max(im.shape[0] for im in imgs)
        W = max(im.shape[1] for im in imgs)
        out = []
        for im in imgs:
            c = np.ones((H, W, im.shape[2]), dtype=im.dtype)
            y0, x0 = (H - im.shape[0]) // 2, (W - im.shape[1]) // 2
            c[y0:y0+im.shape[0], x0:x0+im.shape[1]] = im
            out.append(c)
        return out

    def hstack_gap(panels, gap=6):
        parts = []
        for i, p in enumerate(panels):
            parts.append(p)
            if i < len(panels) - 1:
                parts.append(np.ones((p.shape[0], gap, p.shape[2]), dtype=p.dtype))
        return np.concatenate(parts, axis=1)

    r_strip = hstack_gap(pad_row(panel_imgs, 'R'))
    v_strip = hstack_gap(pad_row(panel_imgs, 'V'))

## 3. Helper functions and LOO correlations

In [ ]:
def shorten_label(lbl: str) -> str:
    return (lbl
        .replace('caudalanteriorcingulate', 'caudACC')
        .replace('rostralanteriorcingulate', 'rostACC')
        .replace('posteriorcingulate', 'PCC')
        .replace('isthmuscingulate', 'isthCing')
        .replace('caudalmiddlefrontal', 'caudMFG')
        .replace('rostralmiddlefrontal', 'rostMFG')
        .replace('superiorfrontal', 'SFG')
        .replace('lateralorbitofrontal', 'latOFC')
        .replace('medialorbitofrontal', 'medOFC')
        .replace('parsopercularis', 'parsOper')
        .replace('parsorbitalis', 'parsOrb')
        .replace('parstriangularis', 'parsTri')
        .replace('superiorparietal', 'SPL')
        .replace('inferiorparietal', 'IPL')
        .replace('superiortemporal', 'STG')
        .replace('middletemporal', 'MTG')
        .replace('inferiortemporal', 'ITG')
        .replace('lateraloccipital', 'latOcc')
        .replace('-lh', ' L')
        .replace('-rh', ' R'))


def select_scatter_labels(r_vals, v_vals, all_labels,
                          n_per_corner=2, max_labels=8):
    """Pick region labels from the four corners of the R̄–V̄ scatter."""
    def _base(lbl):
        for sfx in ('-lh', '-rh'):
            if lbl.endswith(sfx):
                return lbl[:-len(sfx)]
        return lbl

    selected, selected_bases = [], set()
    r_norm = (r_vals - r_vals.min()) / (r_vals.max() - r_vals.min())
    v_norm = (v_vals - v_vals.min()) / (v_vals.max() - v_vals.min())
    corners = [
        r_norm - v_norm,   # high R, low V
        -r_norm + v_norm,  # low R, high V
        r_norm + v_norm,   # high R, high V
        -r_norm - v_norm,  # low R, low V
    ]
    for score in corners:
        added = 0
        for idx in np.argsort(score)[::-1]:
            if added >= n_per_corner or len(selected) >= max_labels:
                break
            base = _base(all_labels[idx])
            if idx not in selected and base not in selected_bases:
                selected.append(idx)
                selected_bases.add(base)
                added += 1
    return selected


# ── Leave-one-out reliability ─────────────────────────────────────
n_s = R.shape[0]
loos_R_sp, loos_V_sp = [], []
loos_R_pe, loos_V_pe = [], []
for s in range(n_s):
    mask = np.ones(n_s, dtype=bool); mask[s] = False
    loo_r = R[mask].mean(0)
    loo_v = V[mask].mean(0)
    loos_R_sp.append(spearmanr(R[s], loo_r).statistic)
    loos_V_sp.append(spearmanr(V[s], loo_v).statistic)
    loos_R_pe.append(pearsonr(R[s],  loo_r).statistic)
    loos_V_pe.append(pearsonr(V[s],  loo_v).statistic)
loos_R_sp = np.array(loos_R_sp)
loos_V_sp = np.array(loos_V_sp)
loos_R_pe = np.array(loos_R_pe)
loos_V_pe = np.array(loos_V_pe)

## 4. Figure parameters

In [ ]:
# ── Scatter label parameters ─────────────────────────────────────
N_PER_CORNER = 2        # candidate labels per scatter corner
MAX_LABELS   = 8        # total label cap
LABEL_FONT   = 5.5      # annotation font size (points)
ARROW_KW     = dict(arrowstyle='-', color='0.45', lw=0.35,
                    shrinkA=0, shrinkB=2)
DROP_LABELS  = {'superiorfrontal-rh'}          # remove from auto-selection
ADD_LABELS   = ['rostralanteriorcingulate-lh']  # force-include

# ── Panel d ───────────────────────────────────────────────────────
LOO_METHOD = 'spearman'   # 'spearman' or 'pearson'

# ── Figure geometry ───────────────────────────────────────────────
FIG_WIDTH  = 7.09    # figure width in inches
SPACER_AB  = 0.08    # vertical gap between brain rows a and b (inches)
SPACER_BC  = 0.42    # vertical gap between row b and bottom panels (inches)
BOTTOM_H   = 2.3     # height of bottom panel row (inches)
BOT_RATIO  = [1.1, 0.9]   # width ratio panel c : d
BOT_WSPACE = 0.42    # horizontal gap between panels c and d

## 5. Generate and save figure

In [ ]:
# ── Scatter label selection ──────────────────────────────────────
lbl_idx = select_scatter_labels(R_mean, V_mean, labels,
                                n_per_corner=N_PER_CORNER, max_labels=MAX_LABELS)
lbl_idx = [i for i in lbl_idx if labels[i] not in DROP_LABELS]
for add_lbl in ADD_LABELS:
    add_i = labels.index(add_lbl)
    if add_i not in lbl_idx:
        lbl_idx.append(add_i)
lbl_xs    = np.array([R_mean[i] for i in lbl_idx])
lbl_ys    = np.array([V_mean[i] for i in lbl_idx])
lbl_names = [shorten_label(labels[i]) for i in lbl_idx]

# ── LOO data for panel d ─────────────────────────────────────────
if LOO_METHOD == 'spearman':
    loo_R, loo_V = loos_R_sp, loos_V_sp
    loo_label = r'Spearman $\rho$ with' + '\nleave-one-out mean'
else:
    loo_R, loo_V = loos_R_pe, loos_V_pe
    loo_label = r'Pearson $r$ with' + '\nleave-one-out mean'

# ── Figure layout ────────────────────────────────────────────────
brain_row_h = FIG_WIDTH * 0.93 * (r_strip.shape[0] / r_strip.shape[1])
fig_height  = 2 * brain_row_h + SPACER_AB + SPACER_BC + BOTTOM_H

fig = plt.figure(figsize=(FIG_WIDTH, fig_height))
gs_main = gridspec.GridSpec(
    5, 1, figure=fig,
    height_ratios=[brain_row_h, SPACER_AB, brain_row_h, SPACER_BC, BOTTOM_H],
    hspace=0)

# ── Panel a: R̄ maps ──────────────────────────────────────────────
gs_a = gridspec.GridSpecFromSubplotSpec(
    1, 2, subplot_spec=gs_main[0], width_ratios=[1, 0.025], wspace=0.02)
ax_a = fig.add_subplot(gs_a[0, 0])
ax_a.imshow(r_strip, aspect='auto'); ax_a.axis('off')
cb_a = fig.colorbar(
    ScalarMappable(norm=Normalize(r_vmin, r_vmax), cmap=cmap),
    cax=fig.add_subplot(gs_a[0, 1]))
cb_a.set_label(r'$\bar{r}$', fontsize=8, labelpad=3)
cb_a.ax.tick_params(labelsize=6)
cb_a.locator = MaxNLocator(nbins=5); cb_a.update_ticks()

# ── Panel b: V̄ maps ──────────────────────────────────────────────
gs_b = gridspec.GridSpecFromSubplotSpec(
    1, 2, subplot_spec=gs_main[2], width_ratios=[1, 0.025], wspace=0.02)
ax_b = fig.add_subplot(gs_b[0, 0])
ax_b.imshow(v_strip, aspect='auto'); ax_b.axis('off')
cb_b = fig.colorbar(
    ScalarMappable(norm=Normalize(v_vmin, v_vmax), cmap=cmap),
    cax=fig.add_subplot(gs_b[0, 1]))
cb_b.set_label(r'$\bar{v}$', fontsize=8, labelpad=3)
cb_b.ax.tick_params(labelsize=6)
cb_b.locator = MaxNLocator(nbins=5); cb_b.update_ticks()

# ── Bottom row ────────────────────────────────────────────────────
gs_bot = gridspec.GridSpecFromSubplotSpec(
    1, 2, subplot_spec=gs_main[4],
    width_ratios=BOT_RATIO, wspace=BOT_WSPACE)

# ── Panel c: R̄ vs V̄ scatter ──────────────────────────────────────
ax_c = fig.add_subplot(gs_bot[0, 0])
ax_c.scatter(R_mean, V_mean, c='#4393C3', s=18, alpha=0.80,
             edgecolors='white', linewidths=0.35, zorder=3)
xf = np.linspace(R_mean.min(), R_mean.max(), 200)
ax_c.plot(xf, np.polyval(np.polyfit(R_mean, V_mean, 1), xf),
          'k-', lw=0.8, alpha=0.45, zorder=2)
texts_c = []
for xi, yi, nm in zip(lbl_xs, lbl_ys, lbl_names):
    t = ax_c.text(xi, yi, nm, fontsize=LABEL_FONT, ha='center', va='center',
                  path_effects=[pe.withStroke(linewidth=2.0, foreground='white')],
                  zorder=5)
    texts_c.append(t)
adjust_text(texts_c, x=lbl_xs, y=lbl_ys, ax=ax_c,
            force_text=(0.8, 1.0), force_points=(0.5, 0.5),
            expand=(1.3, 1.5), ensure_inside_axes=True)
fig.canvas.draw()
for t, xi, yi in zip(texts_c, lbl_xs, lbl_ys):
    tx, ty = t.get_position()
    ax_c.annotate('', xy=(xi, yi), xytext=(tx, ty),
                  arrowprops=ARROW_KW, zorder=4)
ax_c.set_xlabel(r'Population-mean $\bar{r}$', fontsize=8)
ax_c.set_ylabel(r'Population-mean $\bar{v}$', fontsize=8)
ax_c.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.2f}'))
ax_c.spines['top'].set_visible(False)
ax_c.spines['right'].set_visible(False)
ax_c.tick_params(direction='out', length=3)
ax_c.grid(True, alpha=0.15, linewidth=0.5)

# ── Panel d: LOO reliability violin ──────────────────────────────
ax_d = fig.add_subplot(gs_bot[0, 1])
rng = np.random.default_rng(42)
col_R, col_V = '#D6604D', '#4393C3'
for xi, vals, color in [(0, loo_R, col_R), (1, loo_V, col_V)]:
    vp = ax_d.violinplot(vals, positions=[xi], widths=0.55,
                         showmedians=False, showextrema=False)
    for body in vp['bodies']:
        body.set_facecolor(color); body.set_alpha(0.40); body.set_edgecolor('none')
    jx = rng.uniform(-0.14, 0.14, len(vals))
    ax_d.scatter(xi + jx, vals, s=9, color=color, alpha=0.70,
                 edgecolors='white', linewidths=0.3, zorder=3)
    ax_d.plot([xi - 0.20, xi + 0.20], [np.median(vals)] * 2,
              color='black', lw=1.4, zorder=4)
ax_d.axhline(0, color='0.45', lw=0.8, ls='--', zorder=1)
ax_d.set_xticks([0, 1])
ax_d.set_xticklabels([r'$\bar{r}$', r'$\bar{v}$'], fontsize=9)
ax_d.set_ylabel(loo_label, fontsize=7)
ax_d.set_xlim(-0.6, 1.6)
ax_d.spines['top'].set_visible(False)
ax_d.spines['right'].set_visible(False)
ax_d.tick_params(direction='out', length=3)
ax_d.grid(True, axis='y', alpha=0.15, linewidth=0.5)

# ── Panel labels ──────────────────────────────────────────────────
fig.canvas.draw()
lkw = dict(fontsize=11, fontweight='bold', ha='right', va='top',
           transform=fig.transFigure)
bb_a = ax_a.get_position()
bb_b = ax_b.get_position()
bb_c = ax_c.get_position()
bb_d = ax_d.get_position()
x_brain_left = bb_a.x0 - 0.008
fig.text(x_brain_left, bb_a.y1 + 0.005, 'a', **lkw)
fig.text(x_brain_left, bb_b.y1 + 0.005, 'b', **lkw)
fig.text(bb_c.x0 - 0.02, bb_c.y1 + 0.02, 'c', **lkw)
fig.text(bb_d.x0 - 0.02, bb_d.y1 + 0.02, 'd', **lkw)
view_labels = ['L lateral', 'L medial', 'R medial', 'R lateral']
for vi, vl in enumerate(view_labels):
    fig.text(bb_a.x0 + (bb_a.x1 - bb_a.x0) * (vi + 0.5) / 4,
             bb_a.y1 + 0.012, vl, fontsize=6.5, ha='center', va='bottom',
             transform=fig.transFigure, color='0.35')

# ── Save ──────────────────────────────────────────────────────────
out_dir = project_root / 'figures'
out_dir.mkdir(parents=True, exist_ok=True)
for ext in ['pdf', 'png']:
    out_path = out_dir / f'fig_cortical_atlas_maps_{LOO_METHOD}.{ext}'
    fig.savefig(out_path, dpi=300, facecolor='white',
                bbox_inches='tight', pad_inches=0.03)
    print(f'Saved: {out_path}')
plt.show()